In [1]:
%cd /Users/amarmesic/Documents/tudelft/thesis/DNANet

/Users/amarmesic/Documents/tudelft/thesis/DNANet


/Users/amarmesic/miniconda3/envs/dnanet/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
batch_used = 1

In [3]:
# import train
# train.run(
#     "dnanet_rd.yaml",
#     "unet_advanced.yaml",
#     "training_config.yaml",
#     split=batch_used,
#     validation_config=0.1,
#     checkpoint_dir="output/ProvedIt_best_unet/20250926_142826/checkpoint",
#     output_dir="output/result"
#     seed=0
# )

In [4]:
import evaluate

results = evaluate.run(
    data_config="dnanet_rd.yaml",
    model_config="unet.yaml",
    evaluation_config="segmentation.yaml",
    checkpoint_dir="resources/model/current_best_unet",
    split=batch_used,
    seed=0,
    save_preds=False
)
results

2025-10-07 17:31:04 INFO     Logs will be written to output/evaluate_2025-10-07_17-31-04/log_evaluation.txt
2025-10-07 17:31:04 INFO     Loading model...
2025-10-07 17:31:05 INFO     Loading dataset...
2025-10-07 17:31:05 INFO     Loading data from file
2025-10-07 17:31:05 INFO     Walking directory to retrieve all hid files...
Processing folders: 100%|██████████| 20/20 [00:00<00:00, 2146.19it/s]
2025-10-07 17:31:05 INFO     Found 350 HIDs
2025-10-07 17:31:05 INFO     Found 0 HIDs without ladder
2025-10-07 17:31:05 INFO     Removed 0 HIDs without annotation
Loading data from resources/data/2p_5p_Dataset_NFI/Raw data .HID files:  90%|█████████ | 315/350 [00:20<00:01, 17.94it/s]2025-10-07 17:31:25 WARNING  Skipping image: Missing data (resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 5/Inj7 2017-05-12-16-25-53-867/5E4_C08_08.hid)
Skipping image: Missing data (resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 5/Inj7 2017-05-12-16-25-53-867/5E4_C08_08

KeyboardInterrupt: 

In [11]:
import train
import evaluate
import neptune
import numpy as np
import json

all_results = []

for batch_used in range(1, 7):  # 1 to 6 inclusive
    for seed in range(3):       # 0, 1, 2
        # Train
        train.run(
            "dnanet_rd.yaml",
            "unet_advanced.yaml",
            "training_config.yaml",
            split=batch_used,
            validation_config=0.1,
            checkpoint_dir="output/ProvedIt_best_unet/20250926_142826/checkpoint",
            output_dir=f"output/result-batch-{batch_used}-seed-{seed}",
            seed=seed
        )
        # Evaluate
        results = evaluate.run(
            data_config="dnanet_rd.yaml",
            model_config="unet_advanced.yaml",
            evaluation_config="segmentation.yaml",
            checkpoint_dir=f"output/result-batch-{batch_used}-seed-{seed}",
            split=batch_used,
            seed=seed,
        )
        if isinstance(results, str):
            results = json.loads(results)
        all_results.append(results)

        # Neptune logging
        run = neptune.init_run(
            name=f"DNANet的长-1正真:0正假-考试倍{batch_used}-预：训练,缩放-种{seed}",
            project="amar-mesic/dna-thesis",
            api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
        )
        run["test/pixel_f1"] = float(f"{results['pixel_f1_score']:.4g}")
        run["test/pixel_precision"] = float(f"{results['pixel_precision']:.4g}")
        run["test/pixel_recall"] = float(f"{results['pixel_recall']:.4g}")
        run["test/allele_f1"] = float(f"{results['allele_f1_score']:.4g}")
        run["test/allele_precision"] = float(f"{results['allele_precision']:.4g}")
        run["test/allele_recall"] = float(f"{results['allele_recall']:.4g}")
        meta = {
            "experiment": "Cross-kit-FineTune",
            "model": "Amar-DNANet_Advanced",
            "dataset": "PT:Synth+ProvedIt-FT:NFI_R&D",
            "fold": batch_used,
            "seed": seed,
        }
        for k, v in meta.items():
            run[f'meta/{k}'] = v
        run.stop()

# Compute mean and std for each metric
metrics = ["pixel_f1_score", "pixel_precision", "pixel_recall", "allele_f1_score", "allele_precision", "allele_recall"]
summary = {}
for metric in metrics:
    values = [r[metric] for r in all_results]
    summary[metric] = {
        "mean": np.mean(values),
        "std": np.std(values)
    }

print(summary)

2025-10-03 12:14:26 INFO     Logs will be written to output/result-batch-1-seed-0/log_training.txt
2025-10-03 12:14:26 INFO     Loading model...
2025-10-03 12:14:26 INFO     Loading previous model checkpoint from output/ProvedIt_best_unet/20250926_142826/checkpoint
2025-10-03 12:14:26 INFO     Loading dataset...
2025-10-03 12:14:26 INFO     Loading data from file
2025-10-03 12:14:26 INFO     Walking directory to retrieve all hid files...
Processing folders: 100%|██████████| 20/20 [00:00<00:00, 4794.86it/s]
2025-10-03 12:14:26 INFO     Found 350 HIDs
2025-10-03 12:14:26 INFO     Found 0 HIDs without ladder
2025-10-03 12:14:26 INFO     Removed 0 HIDs without annotation
Loading data from resources/data/2p_5p_Dataset_NFI/Raw data .HID files:  90%|████████▉ | 314/350 [00:17<00:01, 21.54it/s]2025-10-03 12:14:43 WARNING  Skipping image: Missing data (resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 5/Inj7 2017-05-12-16-25-53-867/5E4_C08_08.hid)
Skipping image: Missing data

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-648
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 18 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 18 operations synced, thanks for waiting!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-648/metadata


2025-10-03 12:17:49 INFO     Logs will be written to output/result-batch-1-seed-1/log_training.txt
2025-10-03 12:17:49 INFO     Loading model...
2025-10-03 12:17:49 INFO     Loading previous model checkpoint from output/ProvedIt_best_unet/20250926_142826/checkpoint
2025-10-03 12:17:49 INFO     Loading dataset...
2025-10-03 12:17:49 INFO     Loading data from file
2025-10-03 12:17:50 INFO     Walking directory to retrieve all hid files...
Processing folders: 100%|██████████| 20/20 [00:00<00:00, 6251.76it/s]
2025-10-03 12:17:50 INFO     Found 350 HIDs
2025-10-03 12:17:50 INFO     Found 0 HIDs without ladder
2025-10-03 12:17:50 INFO     Removed 0 HIDs without annotation
Loading data from resources/data/2p_5p_Dataset_NFI/Raw data .HID files:  90%|█████████ | 315/350 [00:17<00:02, 17.10it/s]2025-10-03 12:18:07 WARNING  Skipping image: Missing data (resources/data/2p_5p_Dataset_NFI/Raw data .HID files/Mixture dataset 5/Inj7 2017-05-12-16-25-53-867/5E4_C08_08.hid)
Skipping image: Missing data

: 

: 